# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

**Student ID:** 2240004223

Completed solution using the supplied Yelp review dataset.

## Setup



In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "torch": "torch", "gensim": "gensim"}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [ ]:
import random
import re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

candidate_paths = [
    Path("07-yelp-dataset.txt"),
    Path("upload/07-yelp-dataset.txt"),
    Path("/content/07-yelp-dataset.txt")
]
dataset_path = next((path for path in candidate_paths if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Place 07-yelp-dataset.txt beside this notebook or upload it to /content in Colab.")
df = pd.read_csv(dataset_path, sep="\t", names=["review", "label"])
assert df["review"].notna().all()
assert df["label"].isin([0, 1]).all()
print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [ ]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))
pd.DataFrame({
    "Training": y_train.value_counts().sort_index(),
    "Testing": y_test.value_counts().sort_index()
}).rename(index={0: "Negative", 1: "Positive"})

Training reviews: 800
Testing reviews: 200


,Training,Testing
label,,
Negative,400,100
Positive,400,100


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [ ]:
X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)
print("Training tensor shape:", tuple(X_train_tensor.shape))
print("Testing tensor shape:", tuple(X_test_tensor.shape))

Training tensor shape: (800, 1000)
Testing tensor shape: (200, 1000)


## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [ ]:
torch.manual_seed(seed)
input_size = X_train_tensor.shape[1]
model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [ ]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [ ]:
epochs = 10
tfidf_training_history = []
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    tfidf_training_history.append({"Epoch": epoch + 1, "Loss": loss.item()})
    print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 1/10 - Loss: 0.6931
Epoch 2/10 - Loss: 0.6869
Epoch 3/10 - Loss: 0.6726
Epoch 4/10 - Loss: 0.6495
Epoch 5/10 - Loss: 0.6166
Epoch 6/10 - Loss: 0.5748
Epoch 7/10 - Loss: 0.5246
Epoch 8/10 - Loss: 0.4650
Epoch 9/10 - Loss: 0.3997
Epoch 10/10 - Loss: 0.3323


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()
accuracy = accuracy_score(y_test, predictions)
print(f"TF-IDF Test Accuracy: {accuracy:.3f}")

TF-IDF Test Accuracy: 0.760


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [ ]:
print("Classification Report:")
print(classification_report(y_test, predictions, labels=[0, 1], target_names=["Negative", "Positive"], zero_division=0))
pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=[0, 1]),
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.68      0.74       100
    Positive       0.72      0.84      0.78       100

    accuracy                           0.76       200
   macro avg       0.77      0.76      0.76       200
weighted avg       0.77      0.76      0.76       200



,Predicted Negative,Predicted Positive
Actual Negative,68,32
Actual Positive,16,84


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [ ]:
from gensim.models import Word2Vec

train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]
word2vec_model = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    seed=seed
)

def review_to_vector(tokens, word_vectors):
    known_tokens = [token for token in tokens if token in word_vectors]
    if not known_tokens:
        return np.zeros(word_vectors.vector_size, dtype=np.float32)
    return np.mean(word_vectors[known_tokens], axis=0).astype(np.float32)

X_train_word2vec = np.vstack([
    review_to_vector(tokens, word2vec_model.wv) for tokens in train_tokens
])
X_test_word2vec = np.vstack([
    review_to_vector(tokens, word2vec_model.wv) for tokens in test_tokens
])
X_train_word2vec_tensor = torch.tensor(X_train_word2vec, dtype=torch.float32)
X_test_word2vec_tensor = torch.tensor(X_test_word2vec, dtype=torch.float32)
print("Word2Vec vocabulary size:", len(word2vec_model.wv))
print("Training Word2Vec tensor shape:", tuple(X_train_word2vec_tensor.shape))
print("Testing Word2Vec tensor shape:", tuple(X_test_word2vec_tensor.shape))
print("Test tokens outside the training vocabulary:", sum(
    token not in word2vec_model.wv for tokens in test_tokens for token in tokens
))
assert X_train_word2vec_tensor.shape == (len(X_train), 200)
assert X_test_word2vec_tensor.shape == (len(X_test), 200)

Word2Vec vocabulary size: 1787
Training Word2Vec tensor shape: (800, 200)
Testing Word2Vec tensor shape: (200, 200)
Test tokens outside the training vocabulary: 281


### Review-vector method

Each review is represented by the mean of its known word embeddings. Word2Vec is trained only on the training reviews to avoid using test data during feature learning. Unseen test words are ignored, and a review with no known words receives a zero vector. A single worker and fixed seed make training reproducible.

## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [ ]:
torch.manual_seed(seed)
word2vec_network = nn.Sequential(
    nn.Linear(200, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
word2vec_loss_function = nn.BCEWithLogitsLoss()
word2vec_optimizer = torch.optim.Adam(word2vec_network.parameters(), lr=0.001)
word2vec_epochs = 15
word2vec_training_history = []
print(word2vec_network)

for epoch in range(word2vec_epochs):
    word2vec_network.train()
    word2vec_optimizer.zero_grad()
    word2vec_logits = word2vec_network(X_train_word2vec_tensor)
    word2vec_loss = word2vec_loss_function(word2vec_logits, y_train_tensor)
    word2vec_loss.backward()
    word2vec_optimizer.step()
    word2vec_training_history.append({"Epoch": epoch + 1, "Loss": word2vec_loss.item()})
    print(f"Epoch {epoch + 1}/{word2vec_epochs} - Loss: {word2vec_loss.item():.4f}")

word2vec_network.eval()
with torch.no_grad():
    word2vec_test_logits = word2vec_network(X_test_word2vec_tensor)
    word2vec_test_probabilities = torch.sigmoid(word2vec_test_logits)
    word2vec_predictions = (word2vec_test_probabilities >= 0.5).int().numpy().flatten()
word2vec_accuracy = accuracy_score(y_test, word2vec_predictions)
print(f"Word2Vec Test Accuracy: {word2vec_accuracy:.3f}")

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)
Epoch 1/15 - Loss: 0.6958
Epoch 2/15 - Loss: 0.6953
Epoch 3/15 - Loss: 0.6949
Epoch 4/15 - Loss: 0.6946
Epoch 5/15 - Loss: 0.6943
Epoch 6/15 - Loss: 0.6941
Epoch 7/15 - Loss: 0.6938
Epoch 8/15 - Loss: 0.6937
Epoch 9/15 - Loss: 0.6935
Epoch 10/15 - Loss: 0.6934
Epoch 11/15 - Loss: 0.6933
Epoch 12/15 - Loss: 0.6933
Epoch 13/15 - Loss: 0.6932
Epoch 14/15 - Loss: 0.6932
Epoch 15/15 - Loss: 0.6932
Word2Vec Test Accuracy: 0.490


## Word2Vec Evaluation and Model Comparison

In [ ]:
print("Word2Vec Classification Report:")
print(classification_report(y_test, word2vec_predictions, labels=[0, 1], target_names=["Negative", "Positive"], zero_division=0))
pd.DataFrame(
    confusion_matrix(y_test, word2vec_predictions, labels=[0, 1]),
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

Word2Vec Classification Report:
              precision    recall  f1-score   support

    Negative       0.49      0.98      0.66       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.49       200
   macro avg       0.25      0.49      0.33       200
weighted avg       0.25      0.49      0.33       200



,Predicted Negative,Predicted Positive
Actual Negative,98,2
Actual Positive,100,0


In [ ]:
comparison = pd.DataFrame({
    "Representation": ["TF-IDF", "Mean Word2Vec"],
    "Input Features": [input_size, 200],
    "Hidden Layers": ["64, 32", "128, 64, 32"],
    "Learning Rate": [0.01, 0.001],
    "Epochs": [epochs, word2vec_epochs],
    "Test Accuracy": [accuracy, word2vec_accuracy]
})
comparison

,Representation,Input Features,Hidden Layers,Learning Rate,Epochs,Test Accuracy
0,TF-IDF,1000,"64, 32",0.010,10,0.76
1,Mean Word2Vec,200,"128, 64, 32",0.001,15,0.49


## Conclusion

Both models were evaluated on the same stratified 20% test split. The comparison table reports measured accuracies, not expected values. Word2Vec trained from scratch on this small corpus can produce weak sentiment features, and mean pooling discards word order. The requested 15-epoch neural-network training is kept unchanged; a low test accuracy is a valid measured result, not evidence of a missing task.